# Databricks → Neo4j: Load

**Run on dedicated compute. Pipeline step 1 of 3 (slide 12: "The Enrichment Pipeline").**

Load Silver into Neo4j. The five Silver tables leave Unity Catalog and become a property graph in Neo4j Aura: `:Account` and `:Merchant` nodes, `TRANSFERRED_TO` relationships between accounts, and `TRANSACTED_WITH` relationships between accounts and merchants. This is the graph that the next notebook (`04_gds_enrichment`) projects and runs algorithms against.

The same records. A different data structure. A graph database doesn't need a starting account ID to search — it takes a pattern and returns every place the pattern appears in the network. That capability is why the connection questions on slide 10 live in the graph layer and not in SQL.

In [ ]:
%pip install graphdatascience --quiet

In [ ]:
dbutils.library.restartPython()

## 0. Configuration

Run after the `%pip install` kernel restart above.

In [ ]:
CATALOG   = "graph-enriched-lakehouse"
SCHEMA    = "graph-enriched-schema"

NEO4J_URI      = dbutils.secrets.get("neo4j-graph-engineering", "uri")
NEO4J_USER     = dbutils.secrets.get("neo4j-graph-engineering", "username")
NEO4J_PASSWORD = dbutils.secrets.get("neo4j-graph-engineering", "password")

# Common Spark Connector options
NEO4J_OPTS = {
    "url":                    NEO4J_URI,
    "authentication.basic.username": NEO4J_USER,
    "authentication.basic.password": NEO4J_PASSWORD,
    "batch.size":             "10000",
}

spark.sql(f"USE CATALOG `{CATALOG}`")
spark.sql(f"USE SCHEMA `{SCHEMA}`")

## 2. Clear Neo4j (idempotent re-runs)

Optional: wipe previous graph so the demo is repeatable.

In [ ]:
from graphdatascience import GraphDataScience

gds = GraphDataScience(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))
gds.run_cypher("MATCH (n) DETACH DELETE n")
print("Neo4j cleared.")

## 3. Write Account Nodes

In [ ]:
from pyspark.sql import functions as F

accounts_df = spark.table(f"`{CATALOG}`.`{SCHEMA}`.accounts")
n_accounts = accounts_df.count()

(accounts_df
 .write
 .format("org.neo4j.spark.DataSource")
 .mode("Append")
 .options(**NEO4J_OPTS)
 .option("labels", ":Account")
 .option("node.keys", "account_id")
 .save()
)

print(f"Wrote {n_accounts} Account nodes")

## 4. Write Merchant Nodes

In [ ]:
merchants_df = spark.table(f"`{CATALOG}`.`{SCHEMA}`.merchants")
n_merchants = merchants_df.count()

(merchants_df
 .write
 .format("org.neo4j.spark.DataSource")
 .mode("Append")
 .options(**NEO4J_OPTS)
 .option("labels", ":Merchant")
 .option("node.keys", "merchant_id")
 .save()
)

print(f"Wrote {n_merchants} Merchant nodes")

## 5. Create Indexes Before Relationship Writes

Uniqueness constraints also create an index. Without them the Spark Connector
does a full node scan per relationship row. Indexes are required for fast
`account_id` and `merchant_id` lookups when writing relationships.

In [ ]:
gds.run_cypher("""
    CREATE CONSTRAINT account_id_unique IF NOT EXISTS
    FOR (a:Account) REQUIRE a.account_id IS UNIQUE
""")
gds.run_cypher("""
    CREATE CONSTRAINT merchant_id_unique IF NOT EXISTS
    FOR (m:Merchant) REQUIRE m.merchant_id IS UNIQUE
""")
print("Indexes ready.")

## 6. Write TRANSACTED_WITH Relationships (Account → Merchant)

In [ ]:
txn_df = spark.table(f"`{CATALOG}`.`{SCHEMA}`.transactions")
n_txns = txn_df.count()

(txn_df
 .write
 .format("org.neo4j.spark.DataSource")
 .mode("Overwrite")
 .options(**NEO4J_OPTS)
 .option("relationship", "TRANSACTED_WITH")
 .option("relationship.save.strategy", "keys")
 .option("relationship.source.labels", ":Account")
 .option("relationship.source.node.keys", "account_id:account_id")
 .option("relationship.target.labels", ":Merchant")
 .option("relationship.target.node.keys", "merchant_id:merchant_id")
 .save()
)

print(f"Wrote {n_txns} TRANSACTED_WITH relationships")

## 7. Write TRANSFERRED_TO Relationships (Account → Account)

In [ ]:
p2p_df = spark.table(f"`{CATALOG}`.`{SCHEMA}`.account_links")
n_p2p = p2p_df.count()

(p2p_df
 .write
 .format("org.neo4j.spark.DataSource")
 .mode("Overwrite")
 .options(**NEO4J_OPTS)
 .option("relationship", "TRANSFERRED_TO")
 .option("relationship.save.strategy", "keys")
 .option("relationship.source.labels", ":Account")
 .option("relationship.source.node.keys", "src_account_id:account_id")
 .option("relationship.target.labels", ":Account")
 .option("relationship.target.node.keys", "dst_account_id:account_id")
 .save()
)

print(f"Wrote {n_p2p} TRANSFERRED_TO relationships")

## 8. Verify Graph in Neo4j

In [ ]:
result = gds.run_cypher("""
    CALL apoc.meta.stats() YIELD nodeCount, relCount, labels, relTypes
    RETURN nodeCount, relCount, labels, relTypes
""")
display(result)

### Quick Counts

In [ ]:
counts = gds.run_cypher("""
    MATCH (a:Account) WITH count(a) AS accounts
    MATCH (m:Merchant) WITH accounts, count(m) AS merchants
    MATCH ()-[t:TRANSACTED_WITH]->() WITH accounts, merchants, count(t) AS txns
    MATCH ()-[p:TRANSFERRED_TO]->() WITH accounts, merchants, txns, count(p) AS p2p
    RETURN accounts, merchants, txns, p2p
""")
print(counts.to_string(index=False))

### Sample: One Account's Neighbourhood

In [ ]:
sample = gds.run_cypher("""
    MATCH (a:Account)-[r]->(target)
    WITH a, type(r) AS rel_type, labels(target)[0] AS target_type, count(*) AS cnt
    RETURN a.account_id AS account, rel_type, target_type, cnt
    ORDER BY cnt DESC
    LIMIT 10
""")
display(spark.createDataFrame(sample))

**What we just did:** 4 write operations pushed our entire Delta Lake dataset into Neo4j
as a typed property graph: no ETL pipeline, no CSV exports, no Cypher `LOAD CSV`.

**Next →** `04_gds_enrichment`: run graph algorithms to extract fraud-signal features.